# 05 — Deployment and Consumption

This notebook covers how to run the full pipeline end-to-end and how to consume results via the Streamlit dashboard.

**Reference:** Géron Ch.2 — *Launch, Monitor, and Maintain Your System*.

> *"Your solution needs to be deployed into production. This means... making sure it is robust and runs reliably."* — Géron Ch.2

## 1. Delivery Architecture

```
data/raw/df_ready.csv
        |
        v
price_elasticity.cli analyze
        |
        v
reports/price_elasticity.csv   <-- main output
reports/elasticity_summary.json
        |
        v
streamlit run app/streamlit_app.py
        |
        v
localhost:8501 (Price Elasticity Dashboard)
```

**Dashboard features:**
1. **Elasticity tab** — product-level scatter plot, elasticity coefficient with CI
2. **Simulation tab** — interactive price slider, projected demand and revenue
3. **Ranking tab** — all products sorted by elasticity, filterable by R²
4. **Cross-elasticity tab** — substitute and complement products (when data available)

## 2. Verify Artifacts

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

paths = {
    'Analysis results': PROJECT_ROOT / 'reports' / 'price_elasticity.csv',
    'Summary metrics': PROJECT_ROOT / 'reports' / 'elasticity_summary.json',
    'Raw data': PROJECT_ROOT / 'data' / 'raw' / 'df_ready.csv',
    'Streamlit app': PROJECT_ROOT / 'app' / 'streamlit_app.py',
    'Config': PROJECT_ROOT / 'configs' / 'project.toml',
    'Cross-price (optional)': PROJECT_ROOT / 'data' / 'raw' / 'df_crossprice_previous.csv',
}

for name, path in paths.items():
    status = 'OK' if path.exists() else 'MISSING'
    size = f'{path.stat().st_size / 1024:.1f} KB' if path.exists() else '—'
    print(f'{status:8s} {name:35s} {size}')

## 3. Run the Full Pipeline (from notebook)

In [ ]:
from price_elasticity.analysis import run_analysis

result = run_analysis(PROJECT_ROOT / 'configs' / 'project.toml')
print(f"Products analyzed: {result['n_products']}")
print(f"Results saved to: {result['results_path']}")

## 4. Preview Results

In [ ]:
results_path = PROJECT_ROOT / 'reports' / 'price_elasticity.csv'
df = pd.read_csv(results_path)

print('=== Most elastic products (discount candidates) ===')
display(
    df.sort_values('price_elasticity')
    [['product', 'price_elasticity', 'r2', 'p_value', 'observations']]
    .head(10)
    .round(3)
)

print('\n=== Summary statistics ===')
display(df[['price_elasticity', 'r2', 'p_value', 'observations']].describe().round(3))

## 5. How to Launch the Streamlit Dashboard

From your terminal (with the virtual environment activated):

```bash
# Make sure the analysis has been run first
PYTHONPATH=src python -m price_elasticity.cli analyze

# Launch the dashboard
PYTHONPATH=src streamlit run app/streamlit_app.py
```

The dashboard opens at **http://localhost:8501**.

The app auto-runs the pipeline if `reports/price_elasticity.csv` is missing — so for most users, just running `streamlit run` is enough.

## 6. One-Click Pipeline Check

The cell below runs the CLI commands that a recruiter/reviewer would use:

In [ ]:
import subprocess
import sys

env = {'PYTHONPATH': str(PROJECT_ROOT / 'src')}
import os
env.update(os.environ)

for cmd in [
    [sys.executable, '-m', 'price_elasticity.cli', 'validate-config'],
    [sys.executable, '-m', 'price_elasticity.cli', 'profile'],
    [sys.executable, '-m', 'price_elasticity.cli', 'analyze'],
]:
    result = subprocess.run(cmd, capture_output=True, text=True, env=env, cwd=str(PROJECT_ROOT))
    print(f'$ {" ".join(cmd[1:])}  → exit {result.returncode}')
    if result.stdout.strip():
        print(result.stdout.strip()[:300])
    if result.returncode != 0 and result.stderr:
        print('STDERR:', result.stderr.strip()[:200])

## 7. Portfolio Notes

**Best screenshots for your portfolio:**

| Screenshot | What it shows |
|------------|---------------|
| Elasticity tab — price/demand scatter | Log-log regression line for a specific product |
| Simulation tab — price slider | Revenue impact simulation with real-time numbers |
| Ranking tab | All products sorted by elasticity with R² filter |
| Notebook 04 — CI chart | Uncertainty quantification — shows statistical rigor |
| Notebook 04 — Revenue sweet spot | Business translation of elasticity concept |